In [50]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn as sk
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn import linear_model, neighbors, svm, tree, ensemble
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold

In [ ]:
df_train = pd.read_csv('regressao/train.csv')
df_train.head()

,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,internet,romantic,famrel,freetime,goout,Dalc,Walc,health,absences,score
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,no,no,4,3,4,1,1,3,6,5.67
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,yes,no,5,3,3,1,1,3,4,5.33
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,yes,no,4,3,2,2,3,3,10,8.33
3,GP,F,15,U,GT3,T,4,2,health,services,...,yes,yes,3,2,2,1,1,5,2,14.67
4,GP,F,16,U,GT3,T,3,3,other,other,...,no,no,4,3,2,1,2,5,4,8.67


**Features Numericas:** age, absenses

**Features Ordinais:** Medu, Fedu, traveltime, studytime, failures, famrel, freetime, goout, Dalc, Walc, health

**Features Nominais:** 
    - Binárias: school, sex, address, famsize, Pstatus
    - Yes/No: schoolsup, famsup, paid, activities, nursery, higher, internet, romantic
    - Multiplas categorias: Mjob, Fjob, reason, guardian

### 1.2) Pré Processamento


In [ ]:
X = df_train.drop(columns = "score")
y = df_train["score"]

numericas = ["age", "absences"]

ordinais = ["Medu","Fedu","traveltime","studytime","failures",
             "famrel","freetime","goout","Dalc","Walc","health"]

nominais  = ["school","sex","address","famsize","Pstatus","Mjob","Fjob",
             "reason","guardian","schoolsup","famsup","paid","activities",
             "nursery","higher","internet","romantic"]

pre = ColumnTransformer([
        ("num", StandardScaler(), numericas + ordinais), # num = numerico
        ("cat", OneHotEncoder(handle_unknown="ignore"), nominais), # cat = categorico
])



Separei as features de acordo com seus tipos.
Por meio do ColumnTransformer, lidei com os tipos mistos dos dados. Com o StandardScaler() coloquei as features numericas e ordinais na mesma escala (por padrão média 0 e desvio 1) para padronização. Como a técnica de regularização escolhida foi o Ridge, foi necessário padronizar os valores para que a régua de penalização dos coeficientes de cada feature seja a mesma.

#### Modelos

In [38]:
modelos = {
    "LinearRegression": linear_model.LinearRegression(),
    "Ridge":            linear_model.Ridge(),
    "Lasso":            linear_model.Lasso(),
    "ElasticNet":       linear_model.ElasticNet(),
    "KNN":              neighbors.KNeighborsRegressor(),
    "SVR":              svm.SVR(),
    "DecisionTree":     tree.DecisionTreeRegressor(random_state=42),
    "RandomForest":     ensemble.RandomForestRegressor(random_state=42),
    "GradientBoosting": ensemble.GradientBoostingRegressor(random_state=42),
}

In [48]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)

for nome, modelo in modelos.items():
    pipe = Pipeline([
        ("pre", pre),
        ("modelo", modelo)
    ])
    rmse = -cross_val_score(pipe, X, y, cv=cv,
                        scoring="neg_root_mean_squared_error")
    print(nome, ":", rmse.mean().round(3), "+/-", rmse.std().round(3), )

LinearRegression : 3.459 +/- 0.111
Ridge : 3.452 +/- 0.104
Lasso : 3.563 +/- 0.265
ElasticNet : 3.498 +/- 0.259
KNN : 3.596 +/- 0.265
SVR : 3.342 +/- 0.245
DecisionTree : 4.661 +/- 0.416
RandomForest : 3.238 +/- 0.174
GradientBoosting : 3.287 +/- 0.052


In [51]:
from sklearn.model_selection import GridSearchCV

pipe = Pipeline([("pre", pre), ("reg", ensemble.GradientBoostingRegressor(random_state=42))])
grade = {
    "reg__n_estimators":  [100, 300],
    "reg__learning_rate": [0.05, 0.1],
    "reg__max_depth":     [2, 3],
}
gs = GridSearchCV(pipe, grade, cv=cv,
                  scoring="neg_root_mean_squared_error", n_jobs=-1)
gs.fit(X, y)
print(gs.best_params_, -gs.best_score_)

{'reg__learning_rate': 0.1, 'reg__max_depth': 2, 'reg__n_estimators': 100} 3.213427944060437
